# 4주차 정리: 파이썬 크롤링 - API 이용

**주제:** 네이버 API, 공공데이터 API 기반 크롤링

## 학습 목표
1. 분석할 데이터를 웹에서 수집하는 크롤링 방법을 학습하고, 실제로 사용할 수 있다.
2. 개발자를 위해 제공되는 API로 웹 데이터 크롤링을 할 수 있다.

### 크롤링
- **정의:** 웹 페이지를 자동으로 탐색하고 정보를 수집하는 과정
    - 크롤러 또는 스파이더라는 프로그램으로 웹 사이트에서 데이터를 추출
- **중요성:** 대규모 데이터 분석을 위해서는 웹 데이터의 수집이 필요하다.
    - 활용 분야: 데이터 분석(대규모 데이터를 수집하여 통계적 또는 예측 분석 수행), 시장 연구(데이터 수집을 통한 경쟁 분석, 소비자 행동, 가격 비교)
- **주요 크롤링 도구의 예:**
    - `Beautiful Soup`: 파이썬 라이브러리. HTML과 XML 파싱(요소 어디까지 내려갈지)에 특화되어 있으며, 간단하고 직관적인 API 제공.
    - `Selenium`: 웹 애플리케이션 테스팅에 주로 사용되나, 자바스크립트 실행이 필요한 동적 웹 페이지 크롤링에도 사용. *ex) 로그인 처리*
> `XML`: *eXtensible Markup Language*. 태그명을 사용자가 정의할 수 있다.<br>
> `HTML`: *HyperText Markup Language*. 태그명이 이미 정의되어 있다.<br>
> `HTTP`: *HyperText Transfer Protocol*. 아스키코드 체계라, 눈으로 볼 수 있다(Human Readable)<br>
> `Web`: HTTP 프로토콜을 통해 HTML 문서를 주고 받는 것이다.

### REST (Representational State Transfer)
- 웹 기반 서비스를 설계할 때의 원칙.
- 자원을 URI로 표현하고, HTTP 메서드로 조작(보내든, 받든)
- **특징**
    - 자원(Resource) 기반: 모든 자원은 고유한 URI로 접근. 텍스트 뿐만 아니라 모든 멀티미디어 데이터.
    - 무상태성(Stateless): 서버는 요청 간 상태를 저장하지 않음. 즉, 동일한 요청 --> 동일한 응답
    - 표준 HTTP 메서드 사용(get, post 등)
> *URI(Uniform Resource Identifier)*: 식별자<br>
> *URL(Uniform Resource Locator)*: 식별자 + 위치<br>
> ex) `github.com/kyeongtaek`: URI<br>
> ex) `https://www.github.com/kyeongtaek`: URI & URL<br><br>
> `https://`: scheme(https로 동작한다)<br>
> `www.github.com`: hostname<br>
> `/kyeongtaek`: path

### HTTP (HyperText Transfer Protocol)
- **정의:** 웹에서 클라이언트(Client)와 서버(Server)가 데이터를 주고받는 규약
- 인터넷이므로, Client-Server 구조

#### 요청(Request)
- Client가 특정 자원에 대해 읽기(GET), 데이터 저장(POST), 수정(PUT/PATCH), 삭제(DELETE)의 조작을 함.
- Header와 Body로 구성.
- Header에 메서드, Path, 프로토콜 버전 등 서버가 해당 요청에 대해 참고할 정보를 담음. 아스키 코드/utf-8 인코딩되어 넘어가기에, Human-readable하다.
#### 응답(Response)
- Server가 Client의 Request에 대해 답변을 함.
- 상태 코드, Header, Body로 구성.
- 상태 코드: 200(OK), 404(Page Not Found), 500(Internal Server Error)
- Header에 프로토콜 버전, 상태 코드, 상태 코드에 해당하는 메시지, 마지막으로 수정된 시각과 타입 및 길이 등 Body에 대한 정보를 담아서, 파싱할 수 있도록 함.

### 예제) 네이버 뉴스 크롤링 프로그램 구성
def main()

1. 검색어 지정
2. 네이버 뉴스 검색
- `getNaverSearch()` --> `json.load(responseDecode)` 반환
    - `getRequestUrl()` --> `Response.read()`
3. 응답 데이터 정리 후 리스트에 저장
- `processResponseData()` --> `jsonResult` 반환
4. 리스트를 JSON 파일로 저장

In [2]:
import urllib.request
import urllib.parse

import datetime
import time
import json

# 주피터노트북의 파이썬 경로와, dotenv가 설치된 파이썬 경로가 달라서 생기는 import 에러를 잡기 위함.
# import sys
# !{sys.executable} -m pip install python-dotenv

import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

In [11]:
# url 접속을 요청하고 응답을 받아서 반환하는 부분

def getRequestUrl(url):
    req = urllib.request.Request(url) # request 객체 생성
    req.add_header("X-NCP-APIGW-API-KEY-ID", client_id) # 헤더 정보 추가
    req.add_header("X-NCP-APIGW-API-KEY", client_secret)
    
    try:
        resp = urllib.request.urlopen(req) # request 객체를 인자로 받아 서버에 요청을 보내고, 성공하면 응답을 담은 HttpResponse 객체를 반환
        if resp.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())

            return resp.read().decode('utf-8') # utf-8 인코딩 되어 있었으니까, body(resp.read)를 utf-8 디코딩
        else:
            print(f"[{datetime.datetime.now()}] Url Request Not Success: {resp.getcode()}")
            
    except Exception as e: # 예외 있어도, 종료가 아닌 예외처리 코드 동작
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))

        return None

In [13]:
# url을 만들고 getRequestUrl(url)을 호출하여 받은 응답 데이터를 dict로 반환하는 부분

def getNaverSearch(node, srcText, page_start, display): # page_start: 검색 시작 위치. display: 출력 건수
    base = "https://naverapihub.apigw.ntruss.com/search/v1"
    node = "/%s" % node
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), page_start, display) # utf-8 인코딩하는 등 파라미터 세팅

    url = base + node + parameters # str이니까 concat

    responseDecode = getRequestUrl(url) # body를 utf-8 디코딩해서 가져옴

    if (responseDecode == None):
        return None
    else:
        return json.loads(responseDecode) # 응답(json 형식)을 dict 자료형으로 변환해 리턴

In [17]:
# dict 형식의 응답 데이터를 필요한 항목만 정리하여 딕셔너리 리스트인 jsonResult 반환

def processResponseData(post, jsonResult, cnt):
    # print(f'type of post: {type(post)}') --> type of post: <class 'dict'>
    
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pDate = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900') # 문자열 포맷을 날짜로 바꿈
    pDate = pDate.strftime('%Y-%m-%d %H:%M:%S') # 지정한 포맷대로 문자열로 바꿈

    jsonResult.append({
        'cnt': cnt,
        'title': title,
        'description': description,
        'org_link': org_link,
        'link': link,
        'pDate': pDate
    })

    return

In [18]:
def main():
    node = 'news' # 크롤링할 대상
    srcText = input('검색어를 입력하세요: ')
    cnt = 0
    jsonResult = [] # 검색 결과를 저장하는 리스트 객체

    jsonResponse = getNaverSearch(node, srcText, 1, 100) # 네이버 응답(json)을 저장하는 객체
    
    try:
        total = jsonResponse['total'] # 총 기사의 개수(네이버가 제공)
    except Exception as e:
        print('Error: %s' % e)
        return

    while ((jsonResponse != None) and (jsonResponse['display'] != 0)): # 결과 존재하고, 이번 요청에서 가져온 데이터가 더 있음(display != 0)
        for post in jsonResponse['items']: # 100개(최대의 경우)의 기사를 하나씩 꺼내서
            cnt += 1
            processResponseData(post, jsonResult, cnt) # 처리(jsonResult 채움)

        start = jsonResponse['start'] + jsonResponse['display'] # 시작 인덱스에, 이번 요청에서 가져온 데이터 개수를 더해 다음 시작 인덱스를 구함
        jsonResponse = getNaverSearch(node, srcText, start, 100) # 다음 검색

    print('전체 검색 : %d 건' % total)

    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding = 'utf-8') as outfile:
        jsonFile = json.dumps(jsonResult, indent = 4, sort_keys = True, ensure_ascii = False) # json 형태로 변환함(json은 문자열이다!)
        outfile.write(jsonFile) # 파일 만듦

    print('가져온 데이터 : %d 건' % (cnt))
    print('%s_naver_%s.json SAVED' % (srcText, node))
main()

검색어를 입력하세요:  커피


[2026-09-29 11:28:31.832935] Url Request Success
[2026-09-29 11:28:32.077940] Url Request Success
[2026-09-29 11:28:32.336475] Url Request Success
[2026-09-29 11:28:32.603057] Url Request Success
[2026-09-29 11:28:32.884570] Url Request Success
[2026-09-29 11:28:33.176195] Url Request Success
[2026-09-29 11:28:33.465980] Url Request Success
[2026-09-29 11:28:33.763575] Url Request Success
[2026-09-29 11:28:34.125575] Url Request Success
[2026-09-29 11:28:34.458506] Url Request Success
HTTP Error 400: Bad Request
[2026-09-29 11:28:34.621289] Error for URL: https://naverapihub.apigw.ntruss.com/search/v1/news?query=%EC%BB%A4%ED%94%BC&start=1001&display=100
전체 검색 : 2432668 건
가져온 데이터 : 1000 건
커피_naver_news.json SAVED


### 예제) 공공데이터포털 크롤링 프로그램 구성
def main()

1. 수집할 국가 코드와 연도 입력
2. 데이터 수집 요청
- `getTourismStatsService()` --> 데이터 리스트 반환
    - `getTourismStatItem()` --> `json.loads()` 반환
        - `getPublicRequestUrl()` --> `response.read()` 반환
3. 파일 저장

In [1]:
import urllib.request
import sys
import pandas as pd
import datetime
import time
import json

# 주피터노트북의 파이썬 경로와, dotenv가 설치된 파이썬 경로가 달라서 생기는 import 에러를 잡기 위함.
# import sys
# !{sys.executable} -m pip install python-dotenv

import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

service_key = os.getenv("SERVICE_KEY")
alt_service_key = os.getenv("ALT_SERVICE_KEY")

In [13]:
# url 접속을 요청하고 응답을 받아서 반환

def getPublicRequestUrl(url):
    req = urllib.request.Request(url) # request 객체 생성
    try:
        resp = urllib.request.urlopen(req) # request 객체를 인자로 받아 서버에 요청을 보내고, 성공하면 응답을 담은 HttpResponse 객체를 반환
        if resp.getcode() == 200: # 네트워크 통신이 정상적으로 이루어지기만 하면 200 반환.
            print("[%s] Url Request Success" % datetime.datetime.now())
            return resp.read().decode('utf-8') # body에 대한 utf-8 디코딩
        else:
            print("[%s] Url Request Not Successful: %d" % (datetime.datetime.now(), resp.getcode()))
            return None
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None
            

In [31]:
# 데이터 요청 URL을 만들고, getRequestUrl()를 호출해서 받은 응답 데이터를 반환

def getTourismStatsItem(yyyymm, nat_cd, ed_cd):
    service_url = "http://openapi.tour.go.kr/openapi/service/EdrcntTourismStatsService/getEdrcntTourismStatsList"

    # 헤더가 아닌, 파라미터처럼 들어감
    parameters = "?_type=json&serviceKey=" + alt_service_key # 26/9/30 - 작동하지 않아 alt_service_key를 이용
    parameters += "&YM=" + yyyymm # 연월(ex. 201201)
    parameters += "&NAT_CD=" + nat_cd # 국가 코드
    parameters += "&ED_CD=" + ed_cd # 출/입국 구분코드(D or E)

    url = service_url + parameters

    retData = getPublicRequestUrl(url) # body만. utf-8 디코딩됨.

    if (retData == None):
        return None
    else:
        return json.loads(retData) # dict 리턴

In [32]:
# 수집 기간 동안 월 단위로 getTourismStatsItem()을 호출해 받은 데이터를 리스트로 묶어 반환

def getTourismStatsService(nat_cd, ed_cd, nStartYear, nEndYear):
    jsonResult = []
    result = []
    natName = ''
    ed = ''
    dataEND = "{0}{1:0>2}".format(str(nEndYear), str(12)) # ex. 202612
    isDataEnd = 0 # 데이터 끝 확인용 flag 초기화
    
    for year in range(nStartYear, nEndYear+1): # startYear부터 endYear까지 반복
        for month in range(1, 13):
            if(isDataEnd == 1): break # 데이터 끝 flag 설정되어 있으면 작업 중지
            yyyymm = "{0}{1:0>2}".format(str(year), str(month)) # ex. 202601
            jsonData = getTourismStatsItem(yyyymm, nat_cd, ed_cd) # 응답을 dict 형태로 가져옴

            if "response" in jsonData and "header" in jsonData["response"]:
                result_code = jsonData["response"]["header"].get("resultCode") # 응답의 결과 코드 확인

                if result_code != 0 and result_code != "0000": # 정상("0000")이 아닌 경우
                    error_msg = jsonData["response"]["header"].get("resultMsg")
                    print(f"API 에러 발생: {error_msg}") # 에러 메시지 출력
                    break
            
            if jsonData['response']['body']['items'] == '': # 정보 없을 시
                isDataEnd = 1 # 데이터 끝 flag 설정
                dataEND = "{0}{1:0>2}".format(str(year), str(month-1)) # 이번달 전까지 성공했으니까
                print("데이터 없음... \n 제공되는 통계 데이터는 %s년 %s월까지입니다." %(str(year), str(month-1)))
                break
            print(json.dumps(jsonData, indent=4, sort_keys=True, ensure_ascii=False)) # 시각화
            natName = jsonData['response']['body']['items']['item']['natKorNm']
            natName = natName.replace(' ', '') # 공백 제거
            num = jsonData['response']['body']['items']['item']['num']
            ed = jsonData['response']['body']['items']['item']['ed']
            print('[ %s_%s : %s ]' % (natName, yyyymm, num))
            print('--------------------')
            jsonResult.append({ # 딕셔너리 리스트가 필요함(이후 json 파일로 만들기 위해)
                'nat_name': natName,
                'nat_cd': nat_cd,
                'yyyymm': yyyymm,
                'visit_cnt': num
            })
            result.append([natName, nat_cd, yyyymm, num]) # 리스트의 리스트가 필요함(이후 dataframe으로 만들기 위해)
    return (jsonResult, result, natName, ed, dataEND)

In [34]:
def main():
    jsonResult = []
    result = []

    print("<< 국내 입국한 외국인의 통계 데이터를 수집합니다. >>")
    nat_cd = input('국가 코드를 입력하세요(중국: 112 / 일본: 130 / 미국: 275) :') # str.
    nStartYear = int(input('데이터를 몇 년부터 수집할까요? :'))
    nEndYear = int(input('데이터를 몇 년까지 수집할까요? :'))
    ed_cd = "E" # E: 방한외래관광객(입국), D: 해외 출국

    jsonResult, result, natName, ed, dataEND = getTourismStatsService(nat_cd, ed_cd, nStartYear, nEndYear)

    if jsonResult == []: # api 에러나 연도 범위 오류의 경우
        return # 파일을 만들지 않고 종료

    # 파일저장 1: json 파일
    with open('./%s_%s_%d_%s.json' % (natName, ed, nStartYear, dataEND), 'w', encoding='utf-8') as outfile:
        jsonFile = json.dumps(jsonResult, indent = 4, sort_keys = True, ensure_ascii = False) # json(문자열)으로 만듦. 4칸 띄워 표현. 여러 개면 오름차순.
        outfile.write(jsonFile)

    # 파일저장 2: csv 파일
    columns = ["입국자국가", "국가코드", "입국연월", "입국자 수"]
    result_df = pd.DataFrame(result, columns = columns)
    result_df.to_csv('./%s_%s_%d_%s.csv' % (natName, ed, nStartYear, dataEND), index = False, encoding = 'cp949')

main()

<< 국내 입국한 외국인의 통계 데이터를 수집합니다. >>


국가 코드를 입력하세요(중국: 112 / 일본: 130 / 미국: 275) : 112
데이터를 몇 년부터 수집할까요? : 2025
데이터를 몇 년까지 수집할까요? : 2025


[2026-09-30 11:34:29.880948] Url Request Success
{
    "response": {
        "body": {
            "items": {
                "item": {
                    "ed": "방한외래관광객",
                    "edCd": "E",
                    "natCd": 112,
                    "natKorNm": "중  국",
                    "num": 364460,
                    "rnum": 1,
                    "ym": 202501
                }
            },
            "numOfRows": 10,
            "pageNo": 1,
            "totalCount": 1
        },
        "header": {
            "resultCode": "0000",
            "resultMsg": "OK"
        }
    }
}
[ 중국_202501 : 364460 ]
--------------------
[2026-09-30 11:34:30.860168] Url Request Success
{
    "response": {
        "body": {
            "items": {
                "item": {
                    "ed": "방한외래관광객",
                    "edCd": "E",
                    "natCd": 112,
                    "natKorNm": "중  국",
                    "num": 340860,
                    "rnum": 1,
   